# Model Training

## Objective

This notebook trains and evaluates baseline machine learning models for automatic support ticket priority classification.

The workflow includes:

- Loading engineered features
- Training multiple baseline classifiers
- Evaluating model performance
- Comparing model performance
- Selecting the best performing model
- Saving the trained production model

In [1]:
import joblib
import pandas as pd

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

## Load Feature Engineering Artifacts

In [2]:
MODELS_DIR = Path("../artifacts")

In [3]:
X_train = joblib.load(MODELS_DIR / "X_train.pkl")
X_test = joblib.load(MODELS_DIR / "X_test.pkl")

y_train = joblib.load(MODELS_DIR / "y_train.pkl")
y_test = joblib.load(MODELS_DIR / "y_test.pkl")

priority_encoder = joblib.load(
    MODELS_DIR / "priority_encoder.pkl"
)

In [4]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(13070, 5988)
(3268, 5988)
(13070,)
(3268,)


## Baseline Model 1 — Logistic Regression

In [5]:
lr = LogisticRegression(
    random_state=42,
    max_iter=3000,
    class_weight="balanced"
)

lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,3000
,multi_class,'deprecated'


In [6]:
lr_predictions = lr.predict(X_test)

In [7]:
accuracy = accuracy_score(
    y_test,
    lr_predictions
)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5845


In [8]:
print(
    classification_report(
        y_test,
        lr_predictions,
        target_names=priority_encoder.classes_
    )
)

              precision    recall  f1-score   support

        high       0.66      0.64      0.65      1269
         low       0.47      0.60      0.52       675
      medium       0.59      0.52      0.55      1324

    accuracy                           0.58      3268
   macro avg       0.57      0.59      0.58      3268
weighted avg       0.59      0.58      0.59      3268



In [9]:
cm = confusion_matrix(
    y_test,
    lr_predictions
)

pd.DataFrame(
    cm,
    index=priority_encoder.classes_,
    columns=priority_encoder.classes_
)

,high,low,medium
high,818,162,289
low,91,406,178
medium,333,305,686


## Save Logistic Regression Model

The trained Logistic Regression model is saved for future inference and deployment.

In [10]:
joblib.dump(
    lr,
    MODELS_DIR / "logistic_regression.pkl"
)

['..\\models\\logistic_regression.pkl']

## Baseline Model Comparison

Performance of all trained baseline models will be summarized in a comparison table.

In [11]:
results = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [accuracy]
})

results

,Model,Accuracy
0,Logistic Regression,0.584455


## Baseline Model 2 — Linear Support Vector Machine

Linear Support Vector Machine (Linear SVM) is one of the strongest classical algorithms for sparse text classification problems.

It is particularly effective with TF-IDF features and often outperforms Logistic Regression on text datasets.

In [12]:
from sklearn.svm import LinearSVC

In [13]:
svm = LinearSVC(
    random_state=42,
    class_weight="balanced"
)

svm.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,42


In [14]:
svm_predictions = svm.predict(X_test)

In [15]:
svm_accuracy = accuracy_score(
    y_test,
    svm_predictions
)

print(f"Accuracy: {svm_accuracy:.4f}")

Accuracy: 0.6209


In [16]:
print(
    classification_report(
        y_test,
        svm_predictions,
        target_names=priority_encoder.classes_
    )
)

              precision    recall  f1-score   support

        high       0.67      0.70      0.68      1269
         low       0.53      0.55      0.54       675
      medium       0.62      0.59      0.60      1324

    accuracy                           0.62      3268
   macro avg       0.61      0.61      0.61      3268
weighted avg       0.62      0.62      0.62      3268



In [17]:
svm_cm = confusion_matrix(
    y_test,
    svm_predictions
)

pd.DataFrame(
    svm_cm,
    index=priority_encoder.classes_,
    columns=priority_encoder.classes_
)

,high,low,medium
high,883,106,280
low,110,368,197
medium,326,220,778


In [18]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy
    ]
})

results.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy
1,Linear SVM,0.620869
0,Logistic Regression,0.584455


## Save Linear SVM Model

The best-performing baseline model is saved for deployment and future inference.

In [19]:
joblib.dump(
    svm,
    MODELS_DIR / "linear_svm.pkl"
)

['..\\models\\linear_svm.pkl']

## Baseline Model 3 — Multinomial Naive Bayes

Multinomial Naive Bayes is a classical probabilistic algorithm designed for text classification.

It is computationally efficient and serves as a strong baseline for TF-IDF and bag-of-words representations.

In [20]:
nb = MultinomialNB()

nb.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [21]:
nb_predictions = nb.predict(X_test)

In [22]:
nb_accuracy = accuracy_score(
    y_test,
    nb_predictions
)

print(f"Accuracy: {nb_accuracy:.4f}")

Accuracy: 0.5086


In [23]:
print(
    classification_report(
        y_test,
        nb_predictions,
        target_names=priority_encoder.classes_
    )
)

              precision    recall  f1-score   support

        high       0.54      0.60      0.56      1269
         low       0.50      0.22      0.31       675
      medium       0.49      0.57      0.52      1324

    accuracy                           0.51      3268
   macro avg       0.51      0.46      0.47      3268
weighted avg       0.51      0.51      0.50      3268



In [24]:
nb_cm = confusion_matrix(
    y_test,
    nb_predictions
)

pd.DataFrame(
    nb_cm,
    index=priority_encoder.classes_,
    columns=priority_encoder.classes_
)

,high,low,medium
high,756,59,454
low,177,151,347
medium,476,93,755


In [25]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Multinomial Naive Bayes"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy,
        nb_accuracy
    ]
})

results = results.sort_values(
    by="Accuracy",
    ascending=False
)

results

,Model,Accuracy
1,Linear SVM,0.620869
0,Logistic Regression,0.584455
2,Multinomial Naive Bayes,0.508568


## Baseline Model 4 — Random Forest

Random Forest is an ensemble learning algorithm that combines multiple decision trees.

Although Random Forest performs well on many structured datasets, it is evaluated here to compare its effectiveness against linear models on sparse TF-IDF text features.

In [26]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [27]:
rf_predictions = rf.predict(X_test)

In [28]:
rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

print(f"Accuracy: {rf_accuracy:.4f}")

Accuracy: 0.7809


In [29]:
print(
    classification_report(
        y_test,
        rf_predictions,
        target_names=priority_encoder.classes_
    )
)

              precision    recall  f1-score   support

        high       0.78      0.84      0.81      1269
         low       0.92      0.57      0.71       675
      medium       0.74      0.83      0.78      1324

    accuracy                           0.78      3268
   macro avg       0.81      0.75      0.77      3268
weighted avg       0.79      0.78      0.78      3268



In [30]:
rf_cm = confusion_matrix(
    y_test,
    rf_predictions
)

pd.DataFrame(
    rf_cm,
    index=priority_encoder.classes_,
    columns=priority_encoder.classes_
)

,high,low,medium
high,1069,11,189
low,96,387,192
medium,207,21,1096


In [31]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Multinomial Naive Bayes",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy,
        nb_accuracy,
        rf_accuracy
    ]
})

results = results.sort_values(
    by="Accuracy",
    ascending=False
)

results

,Model,Accuracy
3,Random Forest,0.780906
1,Linear SVM,0.620869
0,Logistic Regression,0.584455
2,Multinomial Naive Bayes,0.508568


# Save Best Baseline Model

Random Forest achieved the highest validation accuracy among all evaluated baseline models and is selected as the production baseline model.

In [33]:
joblib.dump(
    rf,
    MODELS_DIR / "random_forest.pkl"
)

print("Random Forest model saved successfully.")

Random Forest model saved successfully.
